# 🪄 pmdarima: Automating ARIMA

**pmdarima** (originally known as "pyramid-arima") is a statistical library designed to bring the functionality of R's famous `auto.arima` to Python.

While `statsmodels` is great for *building* the model, it requires you to manually specify the order `(p, d, q)`. **pmdarima** wraps around `statsmodels` to find these optimal parameters for you automatically.

## 1. The Core Problem
Finding the right ARIMA parameters is difficult:
* **$p$ (AR):** Read from PACF plot.
* **$q$ (MA):** Read from ACF plot.
* **$d$ (Diff):** Read from ADF test.

Reading these plots is subjective (a "dark art"). If you get them wrong, your forecast is junk.

## 2. The Solution: `auto_arima()`
This is the flagship function of the library. It performs a **Grid Search** (or a smarter "Stepwise Search") over multiple combinations of $p, d, q$ and seasonal $P, D, Q$ to find the "best" model.

### How does it define "Best"? (The AIC Score)
It uses the **AIC (Akaike Information Criterion)**.
* **AIC** measures the trade-off between **Goodness of Fit** (how well it predicts history) and **Complexity** (how many variables you used).
* **The Rule:** The **Lower** the AIC, the better.
* *Analogy:* It's like Golf—a lower score wins. `auto_arima` tries 50 different models and returns the one with the lowest "score."

---

## 3. Key Parameters of `auto_arima`
When you run the function, these are the knobs you turn:

* **`start_p`, `start_q`**: Where to start the search (usually 0 or 1).
* **`max_p`, `max_q`**: The maximum complexity allowed (to prevent overfitting).
* **`m` (Seasonality)**: The number of periods in a season (e.g., $m=12$ for monthly data). If $m=1$, it assumes non-seasonal.
* **`seasonal=True`**: Whether to fit a SARIMA model (Seasonal ARIMA).
* **`d=None`**: If you leave this empty, `pmdarima` will automatically run the ADF test to determine the best differencing order ($d$) for you!
* **`stepwise=True`**:
    * *True:* Uses the Hyndman-Khandakar algorithm (smart search). It's fast.
    * *False:* Tries **every** possible combination (Grid search). It's slow but exhaustive.

## 4. Other Useful Functions
* **`plot_diagnostics()`**: Automatically plots the 4 standard residual charts (Histogram, Q-Q, Correlogram) to check if the model is valid.
* **`model_selection.train_test_split`**: A splitter specifically designed for time series (respects temporal order, unlike scikit-learn's random split).

---

## ⚠️ A Warning
While `auto_arima` is powerful, it is not magic.
1.  **Garbage In, Garbage Out:** If your data is fundamentally unpredictable (pure noise), `auto_arima` will still give you a model, but it won't work.
2.  **Overfitting:** If you set `max_p` too high (like 10), it might memorize the past but fail to predict the future.

#### Installing PMDARIMA Library:

In [10]:
# Run the below line to Install PMDARIMA:

# !pip install pmdarima --user

#### Using PMDARIMA:

In [2]:
import numpy as np
import pandas as pd
from pmdarima import auto_arima

In [11]:
# To get help on auto_arima function:

# help(auto_arima)

In [4]:
airline = pd.read_csv('airline_passengers.csv', index_col= 0, parse_dates= True)
births = pd.read_csv('DailyTotalFemaleBirths.csv', index_col= 0, parse_dates= True)

In [5]:
airline.head()

,Thousands of Passengers
Month,
1949-01-01,112
1949-02-01,118
1949-03-01,132
1949-04-01,129
1949-05-01,121


In [6]:
births.head()

,Births
Date,
1959-01-01,35
1959-01-02,32
1959-01-03,30
1959-01-04,31
1959-01-05,44


In [7]:
# Setting Index Frequency of Both Datasets:

airline.index.freq = 'MS'
births.index.freq = 'D'

##### Auto Arima on Airline Data:

In [12]:
stepwise_fit = auto_arima(airline['Thousands of Passengers'],
                         start_p= 1,
                         start_q= 1,
                         max_p= 3,
                         max_q= 3,
                         m= 12,
                         seasonal= True,
                         start_P= 0,
                         d= None,
                         D= 1,
                         trace= True,
                         error_action= 'ignore', # we don't want to know if an order does not work
                         suppress_warnings= True, # we don't want convergence warnings
                         stepwise= True ) # set to stepwise

Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,1,1)[12]             : AIC=1022.896, Time=0.39 sec
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=1031.508, Time=0.02 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=1020.393, Time=0.14 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=1021.003, Time=0.17 sec
 ARIMA(1,1,0)(0,1,0)[12]             : AIC=1020.393, Time=0.03 sec
 ARIMA(1,1,0)(2,1,0)[12]             : AIC=1019.239, Time=0.30 sec
 ARIMA(1,1,0)(2,1,1)[12]             : AIC=inf, Time=2.27 sec
 ARIMA(1,1,0)(1,1,1)[12]             : AIC=1020.493, Time=0.32 sec
 ARIMA(0,1,0)(2,1,0)[12]             : AIC=1032.120, Time=0.21 sec
 ARIMA(2,1,0)(2,1,0)[12]             : AIC=1021.120, Time=0.42 sec
 ARIMA(1,1,1)(2,1,0)[12]             : AIC=1021.032, Time=0.50 sec
 ARIMA(0,1,1)(2,1,0)[12]             : AIC=1019.178, Time=0.35 sec
 ARIMA(0,1,1)(1,1,0)[12]             : AIC=1020.425, Time=0.11 sec
 ARIMA(0,1,1)(2,1,1)[12]             : AIC=inf, Time=1.65 sec
 ARIMA(0,1,1)(1,1,1)[12]     

In [13]:
stepwise_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                      SARIMAX Results                                      
===========================================================================================
Dep. Variable:                                   y   No. Observations:                  144
Model:             SARIMAX(0, 1, 1)x(2, 1, [], 12)   Log Likelihood                -505.589
Date:                             Wed, 17 Dec 2025   AIC                           1019.178
Time:                                     19:07:15   BIC                           1030.679
Sample:                                 01-01-1949   HQIC                          1023.851
                                      - 12-01-1960                                         
Covariance Type:                               opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.3634      0.074     -4.945      0.000      -0.508      -0.219
ar.S.L12      -0.1239      0.090     -1.372      0.170      -0.301       0.053
ar.S.L24       0.1911      0.107      1.783      0.075      -0.019       0.401
sigma2       130.4480     15.527      8.402      0.000     100.016     160.880
===================================================================================
Ljung-Box (L1) (Q):                   0.01   Jarque-Bera (JB):                 4.59
Prob(Q):                              0.92   Prob(JB):                         0.10
Heteroskedasticity (H):               2.70   Skew:                             0.15
Prob(H) (two-sided):                  0.00   Kurtosis:                         3.87
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

##### Auto Arima on Births Data:

In [14]:
stepwise_fit = auto_arima(y= births['Births'],
                         start_p= 0,
                         start_q= 0,
                         max_p= 6,
                         max_q = 3,
                         m= 12,
                         seasonal= False,
                         d= None,
                         trace= True,
                         error_action= 'ignore',
                         suppress_warnings= True,
                         stepwise= True)

C:\Users\shail\AppData\Roaming\Python\Python312\site-packages\pmdarima\arima\_validation.py:62: UserWarning: m (12) set for non-seasonal fit. Setting to 0
  warnings.warn("m (%i) set for non-seasonal fit. Setting to 0" % m)


Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=2650.760, Time=0.02 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=2565.234, Time=0.06 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=2463.584, Time=0.09 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=2648.768, Time=0.01 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=2460.154, Time=0.18 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=2461.271, Time=0.25 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=inf, Time=0.48 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=2460.722, Time=0.21 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=2536.154, Time=0.13 sec
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=2462.826, Time=0.68 sec
 ARIMA(1,1,1)(0,0,0)[0]             : AIC=2459.074, Time=0.07 sec
 ARIMA(0,1,1)(0,0,0)[0]             : AIC=2462.221, Time=0.04 sec
 ARIMA(1,1,0)(0,0,0)[0]             : AIC=2563.261, Time=0.03 sec
 ARIMA(2,1,1)(0,0,0)[0]             : AIC=2460.367, Time=0.11 sec
 ARIMA(1,1,2)(0,0,0)[0]             : 

In [15]:
stepwise_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                  365
Model:               SARIMAX(1, 1, 1)   Log Likelihood               -1226.537
Date:                Wed, 17 Dec 2025   AIC                           2459.074
Time:                        19:10:47   BIC                           2470.766
Sample:                    01-01-1959   HQIC                          2463.721
                         - 12-31-1959                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1252      0.060      2.097      0.036       0.008       0.242
ma.L1         -0.9624      0.017    -56.429      0.000      -0.996      -0.929
sigma2        49.1512      3.250     15.122      0.000      42.781      55.522
===================================================================================
Ljung-Box (L1) (Q):                   0.04   Jarque-Bera (JB):                25.33
Prob(Q):                              0.84   Prob(JB):                         0.00
Heteroskedasticity (H):               0.96   Skew:                             0.57
Prob(H) (two-sided):                  0.81   Kurtosis:                         3.60
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

##### Auto Arima on Synthetic Data:

In [18]:
# Generating Dummy Seasonal Data:

dates = pd.date_range(start='2018-01-01', periods=60, freq='ME')
data = np.linspace(10, 50, 60) + 10*np.sin(np.linspace(0, 20, 60)) + np.random.normal(0, 2, 60)
df = pd.Series(data, index=dates)

In [20]:
# Auto Arima:

model = auto_arima(df, 
                      start_p=1, start_q=1,
                      max_p=3, max_q=3, # Limit complexity
                      m=12,             # Seasonality (12 months)
                      start_P=0, seasonal=True,
                      d=None, D=1,      # Let it find 'd', but force 'D=1' (seasonal diff)
                      trace=True,       # Print the progress
                      error_action='ignore',  
                      suppress_warnings=True, 
                      stepwise=True)

Performing stepwise search to minimize aic
 ARIMA(1,0,1)(0,1,1)[12] intercept   : AIC=inf, Time=0.37 sec
 ARIMA(0,0,0)(0,1,0)[12] intercept   : AIC=391.343, Time=0.00 sec
 ARIMA(1,0,0)(1,1,0)[12] intercept   : AIC=304.280, Time=0.11 sec
 ARIMA(0,0,1)(0,1,1)[12] intercept   : AIC=inf, Time=0.33 sec
 ARIMA(0,0,0)(0,1,0)[12]             : AIC=399.641, Time=0.00 sec
 ARIMA(1,0,0)(0,1,0)[12] intercept   : AIC=319.545, Time=0.03 sec
 ARIMA(1,0,0)(2,1,0)[12] intercept   : AIC=290.356, Time=0.58 sec
 ARIMA(1,0,0)(2,1,1)[12] intercept   : AIC=292.038, Time=0.65 sec
 ARIMA(1,0,0)(1,1,1)[12] intercept   : AIC=inf, Time=0.50 sec
 ARIMA(0,0,0)(2,1,0)[12] intercept   : AIC=inf, Time=0.40 sec
 ARIMA(2,0,0)(2,1,0)[12] intercept   : AIC=292.277, Time=0.85 sec
 ARIMA(1,0,1)(2,1,0)[12] intercept   : AIC=292.318, Time=0.74 sec
 ARIMA(0,0,1)(2,1,0)[12] intercept   : AIC=307.318, Time=0.55 sec
 ARIMA(2,0,1)(2,1,0)[12] intercept   : AIC=278.467, Time=1.31 sec
 ARIMA(2,0,1)(1,1,0)[12] intercept   : AIC=inf, T

In [21]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                      SARIMAX Results                                      
===========================================================================================
Dep. Variable:                                   y   No. Observations:                   60
Model:             SARIMAX(2, 0, 1)x(2, 1, [], 12)   Log Likelihood                -132.234
Date:                             Wed, 17 Dec 2025   AIC                            278.467
Time:                                     19:13:40   BIC                            291.566
Sample:                                 01-31-2018   HQIC                           283.417
                                      - 12-31-2022                                         
Covariance Type:                               opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      2.2083      0.783      2.820      0.005       0.674       3.743
ar.L1          1.7855      0.071     25.055      0.000       1.646       1.925
ar.L2         -0.9052      0.076    -11.882      0.000      -1.055      -0.756
ma.L1         -0.9898      2.463     -0.402      0.688      -5.818       3.838
ar.S.L12      -0.9026      0.232     -3.889      0.000      -1.357      -0.448
ar.S.L24      -0.3736      0.257     -1.456      0.145      -0.876       0.129
sigma2        10.0568     24.808      0.405      0.685     -38.566      58.680
===================================================================================
Ljung-Box (L1) (Q):                   7.83   Jarque-Bera (JB):                 0.33
Prob(Q):                              0.01   Prob(JB):                         0.85
Heteroskedasticity (H):               0.53   Skew:                             0.07
Prob(H) (two-sided):                  0.22   Kurtosis:                         3.38
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

# 📊 Decoding the SARIMAX Results Summary

The summary table is divided into three main sections.

## Section 1: Model Fit Statistics (Top Right)
This section tells you **"How good is this model compared to others?"**

| Term | Meaning | **Goal** |
| :--- | :--- | :--- |
| **No. Observations** | How many data points were used. | N/A |
| **Log Likelihood** | How likely is it that this model produced this data? | Higher is better |
| **AIC** | **Akaike Information Criterion.** The "score" of the model. It balances accuracy vs. complexity. | **Lower is better** |
| **BIC** | Bayesian Information Criterion. Similar to AIC but punishes complexity more harshly. | Lower is better |

> **Key Takeaway:** If you are comparing two models manually, just look at the **AIC**. The one with the lower number wins.

---

## Section 2: Coefficients (The Middle Table)
This section tells you **"Which variables actually matter?"**

* **ar.L1 / ma.L1:** These are the Lag terms (AR or MA) chosen by the model.
* **coef:** The weight (importance) of that term.
    * *Example:* If `ar.L1` coef is 0.8, it means "Today = 0.8 * Yesterday".
* **P>|z| (The P-Value):** The reliability of that specific term.
    * **< 0.05:** This term is statistically significant. Keep it.
    * **> 0.05:** This term might be useless noise. If many terms are > 0.05, your model is likely **Overfitted** (too complex).

> **Key Takeaway:** You want all your `P>|z|` values to be close to **0.000**.

---

## Section 3: Residual Diagnostics (The Bottom Table)
This section checks the **Errors** (Residuals). It answers: *"Did the model capture all the patterns, or is there still information left in the noise?"*

| Test | What it checks | **Ideal Result** |
| :--- | :--- | :--- |
| **Ljung-Box (Q)** | **Autocorrelation.** Are the errors correlated with each other? | **Prob(Q) > 0.05** (We want NO correlation) |
| **Jarque-Bera (JB)** | **Normality.** Do the errors look like a normal Bell Curve? | **Prob(JB) > 0.05** (We want Normal distribution) |
| **Heteroskedasticity** | **Variance.** Does the error spread change over time? | **Prob(H) > 0.05** (We want Constant variance) |

> **Warning:** This is the opposite of the coefficients!
> * In Coefficients, we want P < 0.05 (Significant).
> * In Residuals (here), we want **Prob > 0.05** (Not Significant). We want the tests to "Fail to find a pattern" in the errors.

### ⚡ Summary Checklist
1.  **AIC:** Is it low?
2.  **P>|z| (Coefs):** Are they all < 0.05?
3.  **Prob(Q) (Ljung-Box):** Is it > 0.05? (Are errors random?)

If the answer to all three is **YES**, you have a robust model!